In [ ]:
!pip install -qU google-generativeai==0.8.5 google-ai-generativelanguage==0.6.15 langgraph langchain langchain-google-genai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 21.4 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/ksolves/agentic_ai_hackthon_2026_sample_data.git

Cloning into 'agentic_ai_hackthon_2026_sample_data'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15 (delta 2), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 544.94 KiB | 5.04 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [3]:
%cd agentic_ai_hackthon_2026_sample_data
!ls

/content/agentic_ai_hackthon_2026_sample_data
customers.json	    knowledge-base.md  products.json  tickets.json
Hackathon_2026.pdf  orders.json        README.md


In [4]:
!pip install -qU google-generativeai==0.8.5 google-ai-generativelanguage==0.6.15 langgraph langchain langchain-google-genai openai

In [5]:
import os
import getpass
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [6]:
os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter the Gemini API Key")

Enter the Gemini API Key··········


In [7]:
llm = ChatGoogleGenerativeAI(model = "models/gemini-2.5-flash", temperature =0.2)

In [8]:
def get_query(state: dict) -> dict:
    q = input("Ask anything: ")
    state["query"] = q
    return state

In [9]:
def classify_query(state: dict) -> dict:
    prompt = f"""
Classify this query into:
- study
- coding
- general

Query: {state['query']}

Answer only one word
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    category = response.content.strip().lower()

    print("Category:", category)
    state["category"] = category
    return state

In [10]:
def router(state: dict) -> str:
    cat = state["category"]

    if "study" in cat:
        return "study"
    elif "coding" in cat:
        return "coding"
    else:
        return "general"

In [11]:
def study_node(state: dict) -> dict:
    prompt = f"Explain clearly: {state['query']}"
    response = llm.invoke([HumanMessage(content=prompt)])
    state["answer"] = response.content
    return state


def coding_node(state: dict) -> dict:
    prompt = f"Give simple code for: {state['query']}"
    response = llm.invoke([HumanMessage(content=prompt)])
    state["answer"] = response.content
    return state


def general_node(state: dict) -> dict:
    prompt = f"Answer simply: {state['query']}"
    response = llm.invoke([HumanMessage(content=prompt)])
    state["answer"] = response.content
    return state

In [12]:
import json

tickets = [
 {"id":1,"issue":"Login not working","status":"open"},
 {"id":2,"issue":"Payment failed","status":"open"},
 {"id":3,"issue":"Order not delivered","status":"pending"},
 {"id":4,"issue":"App crashing","status":"open"},
 {"id":5,"issue":"Refund not received","status":"pending"},
 {"id":6,"issue":"Wrong product delivered","status":"closed"},
 {"id":7,"issue":"Unable to signup","status":"open"},
 {"id":8,"issue":"Slow app performance","status":"open"},
 {"id":9,"issue":"Coupon not working","status":"closed"},
 {"id":10,"issue":"Order cancelled automatically","status":"pending"},
 {"id":11,"issue":"Account blocked","status":"open"},
 {"id":12,"issue":"Delivery delay","status":"pending"},
 {"id":13,"issue":"App not opening","status":"open"},
 {"id":14,"issue":"Payment deducted twice","status":"open"},
 {"id":15,"issue":"Product missing in order","status":"closed"},
 {"id":16,"issue":"Address not updating","status":"open"},
 {"id":17,"issue":"Refund taking too long","status":"pending"},
 {"id":18,"issue":"Website down","status":"open"},
 {"id":19,"issue":"Order tracking not updating","status":"open"},
 {"id":20,"issue":"Customer care not responding","status":"pending"}
]

with open("tickets.json","w") as f:
    json.dump(tickets,f)

print("tickets.json created ✅")

tickets.json created ✅


In [13]:
def get_open_tickets():
    return [t for t in tickets if t["status"] == "open"]

print(get_open_tickets())

[{'id': 1, 'issue': 'Login not working', 'status': 'open'}, {'id': 2, 'issue': 'Payment failed', 'status': 'open'}, {'id': 4, 'issue': 'App crashing', 'status': 'open'}, {'id': 7, 'issue': 'Unable to signup', 'status': 'open'}, {'id': 8, 'issue': 'Slow app performance', 'status': 'open'}, {'id': 11, 'issue': 'Account blocked', 'status': 'open'}, {'id': 13, 'issue': 'App not opening', 'status': 'open'}, {'id': 14, 'issue': 'Payment deducted twice', 'status': 'open'}, {'id': 16, 'issue': 'Address not updating', 'status': 'open'}, {'id': 18, 'issue': 'Website down', 'status': 'open'}, {'id': 19, 'issue': 'Order tracking not updating', 'status': 'open'}]


In [14]:
import json

with open("tickets.json") as f:
    tickets = json.load(f)

print(tickets[:2])

[{'id': 1, 'issue': 'Login not working', 'status': 'open'}, {'id': 2, 'issue': 'Payment failed', 'status': 'open'}]


In [15]:
def tickets_node(state: dict) -> dict:
    prompt = f"""
You are a support assistant.

Tickets data:
{tickets}

User query: {state['query']}

Answer clearly.
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    state["answer"] = response.content
    return state


In [16]:
def count_status():
    open_t = len([t for t in tickets if t["status"]=="open"])
    pending_t = len([t for t in tickets if t["status"]=="pending"])
    closed_t = len([t for t in tickets if t["status"]=="closed"])

    return {"open":open_t,"pending":pending_t,"closed":closed_t}

print(count_status())

{'open': 11, 'pending': 6, 'closed': 3}


In [17]:
status = count_status()
print(f"Open: {status['open']}")
print(f"Pending: {status['pending']}")
print(f"Closed: {status['closed']}")


Open: 11
Pending: 6
Closed: 3


In [18]:
print(count_status())

{'open': 11, 'pending': 6, 'closed': 3}


In [19]:
def tickets_node(state: dict) -> dict:
    status = count_status()

    state["answer"] = f"""
Tickets Summary:
Open: {status['open']}
Pending: {status['pending']}
Closed: {status['closed']}
"""
    return state

In [20]:
def tickets_node(state: dict) -> dict:
    status = count_status()

    if "open" in state["query"].lower():
        answer = f"Open tickets: {status['open']}"
    elif "pending" in state["query"].lower():
        answer = f"Pending tickets: {status['pending']}"
    else:
        answer = f"""
Open: {status['open']},
Pending: {status['pending']},
Closed: {status['closed']}
"""

    state["answer"] = answer
    return state

In [36]:
builder = StateGraph(dict)

builder.add_node("get_query", get_query)
builder.add_node("classify_query", classify_query)
builder.add_node("study_node", study_node)
builder.add_node("coding_node", coding_node)
builder.add_node("general_node", general_node)
builder.add_node("tickets_node", tickets_node)

builder.set_entry_point("get_query")

builder.add_edge("get_query", "classify_query")

builder.add_conditional_edges(
    "classify_query",
    router,
    {
        "study": "study_node",
        "coding": "coding_node",
        "general": "tickets_node"
    },
)

builder.add_edge("study_node", END)
builder.add_edge("coding_node", END)
builder.add_edge("general_node", END)
builder.add_edge("tickets_node", END)

graph = builder.compile()

In [ ]:
final_state = graph.invoke({})
print("final Output \n")
print(final_state["answer"])